In [6]:
import warnings
from langchain._api import LangChainDeprecationWarning

warnings.simplefilter("ignore", category=LangChainDeprecationWarning)

In [5]:
import os
from dotenv import load_dotenv,find_dotenv
_ = load_dotenv(find_dotenv())

groq_api_key =  os.environ["GROQ_API_KEY"]

In [7]:
from langchain_groq import ChatGroq
llmModel = ChatGroq(temperature=0,model = "llama3-70b-8192")

In [8]:
from langchain_community.tools.tavily_search import TavilySearchResults
search = TavilySearchResults(max_results=2)
search.invoke("Who are the top stars of 2024 EuroCup?")

[{'title': "Europe's Elite: Ranking the Top 10 Stars of Euro 2024 - SportsGrid",
  'url': 'https://www.sportsgrid.com/soccer/article/europes-elite-ranking-the-top-10-stars-of-euro-2024',
  'content': "Europe's Elite: Ranking the Top 10 Stars of Euro 2024 · 1. Kylian Mbappe - France · 2. Virgil van Dijk - Netherlands · 3. Jude Bellingham - England · 4. Harry Kane",
  'score': 0.8839694},
 {'title': 'EURO 2024 player Power Rankings: Who are the top 20 stars?',
  'url': 'https://www.nbcsports.com/soccer/news/euro-2024-player-power-rankings-who-are-the-top-20-stars',
  'content': 'EURO 2024 player Power Rankings\n20. Giorgi Mamardashvili (Georgia)\n19. Khvicha Kvaratskhelia (Georgia)\n18. Hakan Calhanoglu (Turkey)\n17. Lamine Yamal (Spain)\n16. N’Golo Kante (France)\n15. Jude Bellingham (England)\n14. Alvaro Morata (Spain)\n13. Kai Havertz (Germany)\n12. Toni Kroos (Germany)\n11. Ilkay Gundogan (Germany)\n10. Cody Gakpo (Netherlands)\n9. Kevin de Bruyne (Belgium)\n8. William Saliba (France

In [12]:
tools=[search]

In [13]:
llm_with_tools = llmModel.bind_tools(tools)

# Create the agent

In [14]:
from langgraph.prebuilt import create_react_agent

agent_executer = create_react_agent(llmModel, tools)

# Run the agent

In [15]:
from langchain_core.messages import HumanMessage

response = agent_executer.invoke({"messages": [HumanMessage(content="Where is the soccer Eurocup 2024 played?")]})

response["messages"]

[HumanMessage(content='Where is the soccer Eurocup 2024 played?', additional_kwargs={}, response_metadata={}, id='33c41ee3-5749-4784-b45c-0ca9ce1aa3dd'),
 AIMessage(content='The 2024 European Football Championship, commonly referred to as Euro 2024, is scheduled to take place in Germany from June 14 to July 14, 2024.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 1921, 'total_tokens': 1959, 'completion_time': 0.129195391, 'prompt_time': 0.070797512, 'queue_time': -0.257400862, 'total_time': 0.199992903}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_dd4ae1c591', 'finish_reason': 'stop', 'logprobs': None}, id='run-516359d7-57d2-403a-bc27-b87387d4f102-0', usage_metadata={'input_tokens': 1921, 'output_tokens': 38, 'total_tokens': 1959})]

# Adding memory

In [16]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [17]:
agent_executer = create_react_agent(llmModel, tools, checkpointer=memory)

In [18]:
config = {"configurable": {"thread_id": "001"}}

In [20]:
response = agent_executer.invoke({"messages": [HumanMessage(content="Where is the soccer Eurocup 2024 played?")]},config)

In [21]:
response

{'messages': [HumanMessage(content='Where is the soccer Eurocup 2024 played?', additional_kwargs={}, response_metadata={}, id='619443c5-5dc8-4be1-9a33-bf0c9e57e6fe'),
  AIMessage(content='The 2024 European Football Championship, commonly referred to as Euro 2024, is scheduled to take place in Germany from June 14 to July 14, 2024.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 1921, 'total_tokens': 1959, 'completion_time': 0.130839485, 'prompt_time': 0.07124633, 'queue_time': -0.253789613, 'total_time': 0.202085815}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_dd4ae1c591', 'finish_reason': 'stop', 'logprobs': None}, id='run-207257b9-5573-40d6-b9f8-8e686ccab484-0', usage_metadata={'input_tokens': 1921, 'output_tokens': 38, 'total_tokens': 1959})]}

In [22]:
config2 = {"configurable": {"thread_id": "002"}}
response = agent_executer.invoke({"messages": [HumanMessage(content="What was my question previously?")]},config2)
response

{'messages': [HumanMessage(content='What was my question previously?', additional_kwargs={}, response_metadata={}, id='a7db2f51-a6f5-402a-81d5-a16ad5163ad8'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_47jg', 'function': {'arguments': '{"query":"What was my question previously?"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 945, 'total_tokens': 1019, 'completion_time': 0.211428571, 'prompt_time': 0.040800128, 'queue_time': 0.054967062000000004, 'total_time': 0.252228699}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_dd4ae1c591', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-1e1d9dfb-dd35-4b42-bdb5-12d58afa2ad6-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'What was my question previously?'}, 'id': 'call_47jg', 'type': 'tool_call'}], usage_metadata={'input_tokens': 945, 'output_tokens': 74, 'total_tokens': 1019})